https://github.com/FabricioArendTorres/FlowConductor/tree/main

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os
import numpy as np
import matplotlib.pyplot as plt

import torch
from msi.flow_conductor import architecture
from msi.flow_conductor.likelihood_flow import LikelihoodFlow

# import tensorflow as tf
# tf.config.experimental.set_memory_growth(tf.config.list_physical_devices(device_type="GPU")[0], True)
# from msi.gaussian_mixture.likelihood_gmm import LikelihoodGMM
# from msi.gaussian_mixture import architecture
# from deep_lss.utils import configuration

from msi.utils import preprocessing, plotting, diagnostics
from msfm.utils import prior, parameters, files, logger, observation

# load network predictions

# v14

### lensing

In [3]:
# conf = files.load_config("/global/homes/a/athomsen/multiprobe-simulation-forward-model/configs/v14/extended.yaml")
# base_dir = ""

# # mutual info loss ###############################################################################################

# # young-serenity-1089 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/3g0z8qob/overview)
# # first v14 run
# model_dir = "/pscratch/sd/a/athomsen/run_files/v14/extended/lensing/mutual_info/2025-04-19_18-54-31_deepsphere_default"
# # n_steps = 100_000
# # n_steps = 200_000
# # n_steps = 300_000
# n_steps = 400_000

# # fisher info loss ###############################################################################################

# params = ["Om", "s8", "w0", "Aia", "n_Aia", "bta"]
# buzzard_cosmo = {"Om": 0.286, "s8": 0.82, "w0": -1, "Aia": 0.0, "n_Aia": np.nan, "bta": 0.0}

### clustering

In [4]:
# conf = files.load_config("/global/homes/a/athomsen/multiprobe-simulation-forward-model/configs/v14/extended.yaml")
# base_dir = ""

# # mutual info loss ###############################################################################################

# # classic-frost-1096 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/fp2vxm07/overview)
# # first v14 clustering probes run
# model_dir = "/pscratch/sd/a/athomsen/run_files/v14/extended/clustering/mutual_info/2025-05-14_23-10-45_deepsphere_default"
# # n_steps = 80_000
# # n_steps = 160_000
# n_steps = 240_000

# # fisher info loss ###############################################################################################

# params = ["Om", "s8", "w0", "bg1", "bg2", "bg3", "bg4"]
# buzzard_cosmo = {"Om": 0.286, "s8": 0.82, "w0": -1, "bg1": np.nan, "bg2": np.nan, "bg3": np.nan, "bg4": np.nan}

### combined

In [5]:
# conf = files.load_config("/global/homes/a/athomsen/multiprobe-simulation-forward-model/configs/v14/extended.yaml")
# base_dir = ""

# # mutual info loss ###############################################################################################

# # grateful-universe-1093 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/tun9hdvl/overview)
# # mythical-cantina-1094 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/6w6yju8t/overview)
# # first v14 combined probes run
# model_dir = "/pscratch/sd/a/athomsen/run_files/v14/extended/combined/mutual_info/2025-04-30_02-27-42_deepsphere_default"
# # n_steps = 100_000
# # n_steps = 200_000
# # n_steps = 300_000
# n_steps = 400_000

# # fisher info loss ###############################################################################################

# params = ["Om", "s8", "w0", "Aia", "n_Aia", "bta", "bg1", "bg2", "bg3", "bg4"]
# buzzard_cosmo = {"Om": 0.286, "s8": 0.82, "w0": -1, "Aia": 0.0, "n_Aia": np.nan, "bta": 0.0, "bg1": np.nan, "bg2": np.nan, "bg3": np.nan, "bg4": np.nan}

# v15

In [12]:
conf = files.load_config("/global/homes/a/athomsen/multiprobe-simulation-forward-model/configs/v15/extended.yaml")

### lensing

In [7]:
# likely-vortex-1179 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/9d63ista/overview)
model_dir = "/pscratch/sd/a/athomsen/run_files/v15/extended/lensing/mutual_info/default/v2"
# n_steps = 200_000
n_steps = 400_000
# n_steps = 450_000
# n_steps = 600_000

checkpoint_number = n_steps // 10_000
n_z = 4
lensing = True

params = ["Om", "s8", "w0", "Aia", "n_Aia", "bta"]
buzzard_cosmo = {"Om": 0.286, "s8": 0.82, "w0": -1, "Aia": 0.0, "n_Aia": np.nan, "bta": 0.0}

### combined

In [8]:
# # deep-sound-1177 (https://wandb.ai/eth-cosmo/y3-deep-lss/runs/kqj560dy/overview)
# model_dir = "/pscratch/sd/a/athomsen/run_files/v15/extended/combined/mutual_info/default/v1"
# # n_steps = 440_000
# n_steps = 520_000

# checkpoint_number = n_steps // 10_000
# params = ["Om", "s8", "w0", "Aia", "n_Aia", "bta", "bg1", "bg2", "bg3", "bg4"]
# buzzard_cosmo = {"Om": 0.286, "s8": 0.82, "w0": -1, "Aia": 0.0, "n_Aia": np.nan, "bta": 0.0, "bg1": np.nan, "bg2": np.nan, "bg3": np.nan, "bg4": np.nan}
# n_z = 8
# combined = True

### general

In [9]:
base_dir = ""

# dataset
fidu_preds, grid_preds, grid_cosmos, file_dict = preprocessing.get_reshaped_network_preds(
    base_dir,
    model_dir,
    n_steps,
    with_fidu=False,
    # n_params=len(params), 
    # n_perms_per_cosmo=4
)

if params == ["Om", "s8"]:
    grid_cosmos = grid_cosmos[:,:2]
if params == ["Om", "s8", "w0"]:
    grid_cosmos = grid_cosmos[:,:3]

# output directory and file names
out_dir = os.path.join(base_dir, model_dir)

label = f"{n_steps}_steps_likelihood"
# label = f"{n_steps}_steps_posterior"

26-02-20 01:09:29 input_output INF   Loading predictions from /pscratch/sd/a/athomsen/run_files/v15/extended/lensing/mutual_info/default/v2/preds_400000.h5 
26-02-20 01:09:29 input_output INF   Array shapes: 
26-02-20 01:09:29 input_output INF   fiducial/vali/pred = (40000, 12) 
26-02-20 01:09:29 input_output INF   fiducial/vali/i_example = (40000,) 
26-02-20 01:09:29 input_output INF   fiducial/vali/i_noise = (40000,) 
26-02-20 01:09:29 input_output INF   grid/pred          = (2500, 80, 12) 
26-02-20 01:09:29 input_output INF   grid/cosmo         = (2500, 80, 6) 
26-02-20 01:09:29 input_output INF   grid/i_example     = (2500, 80) 
26-02-20 01:09:29 input_output INF   grid/i_noise       = (2500, 80) 
26-02-20 01:09:29 input_output INF   grid/i_sobol       = (2500, 80) 


26-02-20 01:09:29 preprocessin INF   Shapes after concatenation and selection: 
26-02-20 01:09:29 preprocessin INF   grid_preds  = (200000, 12) 
26-02-20 01:09:29 preprocessin INF   grid_cosmos = (200000, 6) 


# likelihood Flow $p(x|\theta)$

### architecture

In [10]:
# # input dimensions
# x_dim = grid_preds.shape[-1]
# theta_dim = grid_cosmos.shape[-1]

# # shared hyperparameters
# context_embedding_dim = 32

# embedding_net = architecture.get_context_embedding_net(
#     context_dim=theta_dim,
#     context_embedding_dim=context_embedding_dim,
#     hidden_dim=64,
#     n_blocks=3,
#     dropout_probability=0.0,
#     use_batch_norm=False,
# )    

# base_dist = architecture.get_normal_dist(
#     feature_dim=x_dim,
# )

# transform = architecture.get_sigmoids_transform(
#     feature_dim=x_dim,
#     context_embedding_dim=context_embedding_dim,
#     n_layers=4,
#     hidden_dim=256,
#     svd_kwargs={},
#     sigmoids_kwargs={
#         "n_sigmoids": 16,
#         "num_blocks": 3,
#         "dropout_probability": 0.0,
#     }
# )


# # transform = architecture.get_lipschitz_transform(
# #     feature_dim=x_dim,
# #     context_embedding_dim=context_embedding_dim,
# #     n_layers=4,
# #     hidden_dim=256,
# #     # hidden_dim=512,
# # )

# model = LikelihoodFlow(
#     params, 
#     conf, 
#     embedding_net=embedding_net,
#     base_dist=base_dist,
#     transform=transform,
#     out_dir=out_dir,
#     # label=label,
#     # label=label + "_" + str(params).replace(" ", "").replace("'", ""),
#     # label=label + "_" + str(params).replace(" ", "").replace("'", "") + "_lipschitz",
#     # label=label + "_small_sigmoid",
#     # label=label + "_lipschitz",
#     # label=label + "_sigmoid",
#     label=label + "_sigmoid_test_v2",
#     # label=label + "_lipschitz_underfit",
#     # label=label + "
#     # load_existing=False,
#     load_existing=True,
#     # device="cpu",
#     torch_seed=7,
# )

In [13]:
# # input dimensions
# x_dim = grid_preds.shape[-1]
# theta_dim = grid_cosmos.shape[-1]

# # shared hyperparameters
# context_embedding_dim = 32

# embedding_net = architecture.get_context_embedding_net(
#     context_dim=theta_dim,
#     context_embedding_dim=context_embedding_dim,
#     hidden_dim=64,
#     n_blocks=3,
#     dropout_probability=0.0,
#     use_batch_norm=False,
# )    

# base_dist = architecture.get_normal_dist(
#     feature_dim=x_dim,
# )

# transform = architecture.get_sigmoids_transform(
#     feature_dim=x_dim,
#     context_embedding_dim=context_embedding_dim,
#     n_layers=4,
#     hidden_dim=256,
#     svd_kwargs={},
#     sigmoids_kwargs={
#         "n_sigmoids": 16,
#         "num_blocks": 3,
#         "dropout_probability": 0.0,
#     }
# )

# model = LikelihoodFlow(
#     params, 
#     conf, 
#     embedding_net=embedding_net,
#     base_dist=base_dist,
#     transform=transform,
#     feature_dim=grid_preds.shape[-1],
#     out_dir=out_dir,
#     # label=label,
#     # label=label + "_default",
#     label=label + "_sigmoid",
#     load_existing=True,
# )

26-02-20 01:11:55 likelihood_b INF   Set up the model directory /pscratch/sd/a/athomsen/run_files/v15/extended/lensing/mutual_info/default/v2/400000_steps_likelihood_sigmoid/likelihood_flow 


TypeError: super(type, obj): obj must be an instance or subtype of type

### training

In [ ]:
# n_cosmos = file_dict["grid/pred"].shape[0]
# n_examples = grid_preds.shape[0]
# # such that GPU utilization is maximized, but not larger
# batch_size = 4 * n_cosmos
# print(f"batch_size = {batch_size} for {n_examples / batch_size} steps per epoch")

# # default to train from scratch with 4 permutations per grid point
# model.fit(
#     x=grid_preds,
#     theta=grid_cosmos,
#     n_epochs=100,
#     # dataset
#     batch_size=batch_size,
#     vali_split=0.1,
#     # optimizer
#     learning_rate=1e-3,
#     weight_decay=0.0,
#     clip_by_global_norm=1.0,
#     # scheduler
#     scheduler_type="cosine",
#     scheduler_kwargs={"eta_min": 1e-6},
#     # early stopping
#     n_patience_epochs=None,
#     min_delta=1e-5,
#     save_model=True,
# )

In [ ]:
# # v2
# n_cosmos = file_dict["grid/pred"].shape[0]
# n_examples = grid_preds.shape[0]
# # such that GPU utilization is maximized, but not larger
# batch_size = 4 * n_cosmos
# print(f"batch_size = {batch_size} for {n_examples / batch_size} steps per epoch")

# # default to train from scratch with 4 permutations per grid point
# model.fit(
#     x=grid_preds,
#     theta=grid_cosmos,
#     n_epochs=100,
#     # dataset
#     batch_size=batch_size,
#     vali_split=0.1,
#     # optimizer
#     learning_rate=1e-3,
#     weight_decay=1e-3,
#     clip_by_global_norm=1.0,
#     # scheduler
#     scheduler_type="cosine",
#     scheduler_kwargs={"eta_min": 1e-5},
#     save_model=True,
# )

In [ ]:
# n_cosmos = file_dict["grid/pred"].shape[0]
# n_examples = grid_preds.shape[0]
# # such that GPU utilization is maximized, but not larger
# batch_size = 4 * n_cosmos
# print(f"batch_size = {batch_size} for {n_examples / batch_size} steps per epoch")

# # default to train from scratch with 4 permutations per grid point
# model.fit(
#     x=grid_preds,
#     theta=grid_cosmos,
#     n_epochs=100,
#     # dataset
#     batch_size=batch_size,
#     vali_split=0.1,
#     # optimizer
#     learning_rate=1e-3,
#     weight_decay=1e-3,
#     clip_by_global_norm=1.0,
#     # scheduler
#     scheduler_type="cosine",
#     scheduler_kwargs={"eta_min": 1e-5},
#     save_model=True,
# )

# posterior Flow $p(\theta|x)$

### architecture

In [ ]:
# # input dimensions
# x_dim = grid_preds.shape[-1]
# theta_dim = grid_cosmos.shape[-1]

# # shared hyperparameters
# context_embedding_dim = 32

# embedding_net = architecture.get_context_embedding_net(
#     context_dim=x_dim,
#     context_embedding_dim=context_embedding_dim,
#     hidden_dim=64,
#     n_blocks=3,
#     dropout_probability=0.0,
#     use_batch_norm=False,
# )    

# base_dist = architecture.get_normal_dist(
#     feature_dim=theta_dim,
# )

# transform = architecture.get_sigmoids_transform(
#     feature_dim=theta_dim,
#     context_embedding_dim=context_embedding_dim,
#     n_layers=4,
#     hidden_dim=256,
#     svd_kwargs={},
#     sigmoids_kwargs={
#         "n_sigmoids": 16,
#         "num_blocks": 3,
#         "dropout_probability": 0.0,
#     }
# )

# # transform = architecture.get_lipschitz_transform(
# #     feature_dim=theta_dim,
# #     context_embedding_dim=context_embedding_dim,
# #     n_layers=4,
# #     hidden_dim=256,
# # )

# model = LikelihoodFlow(
#     params, 
#     conf, 
#     embedding_net=embedding_net,
#     base_dist=base_dist,
#     transform=transform,
#     out_dir=out_dir, 
#     # label=label,
#     label=label + "_sigmoid",
#     # label=label + "_wide_lipschitz",
#     # label=label + "_tight",
#     # label=label + "_wide",
#     load_existing=False,
#     # load_existing=True,
# )

### training

In [ ]:
# # only wide prior
# i_in_wide = grid_cosmos.shape[0]//2
# grid_cosmos_wide = grid_cosmos[:i_in_wide]
# grid_preds_wide = grid_preds[:i_in_wide]

# # # only tight prior
# # grid_cosmos = grid_cosmos[i_in_wide:]
# # grid_preds = grid_preds[i_in_wide:]

# n_cosmos = file_dict["grid/pred"].shape[0]
# n_examples = grid_preds.shape[0]
# # such that GPU utilization is maximized, but not larger
# batch_size = 4 * n_cosmos
# print(f"batch_size = {batch_size} for {n_examples / batch_size} steps per epoch")

# # default to train from scratch with 4 permutations per grid point
# model.fit(
#     x=grid_cosmos_wide,
#     theta=grid_preds_wide,
#     n_epochs=100,
#     # n_epochs=300,
#     # dataset
#     batch_size=batch_size,
#     vali_split=0.1,
#     # optimizer
#     learning_rate=1e-3,
#     weight_decay=0.0,
#     clip_by_global_norm=1.0,
#     # scheduler
#     scheduler_type="cosine",
#     scheduler_kwargs={"eta_min": 1e-6},
#     # early stopping
#     n_patience_epochs=None,
#     min_delta=1e-5,
#     save_model=True,
# )

# Gaussian Mixture Model $p(x|\theta)$

### architecture

In [ ]:
# layers = architecture.get_gmm_layers(
#     n_x=grid_preds.shape[-1],
#     n_theta=grid_cosmos.shape[1],
#     n_gaussians=8,
#     n_units=512,
#     n_layers=8,
#     activation="relu",
#     dropout_rate=0.1,
# )

# model = LikelihoodGMM(
#     params, 
#     conf,
#     layers=layers,
#     out_dir=out_dir, 
#     label=label,
#     load_existing=False,
# )

### training

In [ ]:
# n_cosmos = file_dict["grid/pred"].shape[0]
# n_examples = grid_preds.shape[0]
# # such that GPU utilization is maximized, but not larger
# batch_size = 8 * n_cosmos
# print(f"batch_size = {batch_size} for {n_examples / batch_size} steps per epoch")

# model.fit(
#     x=grid_preds,
#     theta=grid_cosmos,
#     n_epochs=1_000,
#     # dataset
#     batch_size=batch_size,
#     vali_split=0.1,
#     # optimizer
#     learning_rate=1e-3,
#     weight_decay=0.0,
#     clip_by_global_norm=1.0,
#     # schedule
#     # scheduler_kwargs={"factor": 0.75, "patience": 20, "cooldown": 10, "min_lr": 1e-6},
#     scheduler_kwargs={"factor": 0.8, "patience": 10, "cooldown": 5, "min_lr": 1e-6},
#     # scheduler_kwargs={},
#     # early stopping
#     n_patience_epochs=100,
#     min_delta=1e-5,
#     save_model=True,
# )

### convergence tests

In [ ]:
# grid_preds_sample = model.plot_diagnostics(
#     grid_preds_true=grid_preds,
#     grid_cosmos=grid_cosmos,
#     n_samples=100,
#     n_cosmos=1000,
#     do_eecp=True,
#     do_tarp=True,
#     tarp_kwargs = {"n_bootstrap": 100, "n_alpha_bins": 20}
# )

In [ ]:
# %%time
# n_cosmos = 1000
# n_samples = 100
# batch_size = 1000

# grid_preds_true = grid_preds.copy()
# if n_cosmos is not None:
#     print(f"Selecting {n_cosmos} random cosmologies")
#     random_indices = np.random.choice(grid_preds_true.shape[0], n_cosmos, replace=False)
#     grid_preds_true = grid_preds_true[random_indices]
#     grid_cosmos_diagnostics = grid_cosmos[random_indices]

# print(f"Drawing samples from the likelihood")
# grid_preds_sample = model.sample_likelihood(
#     grid_cosmos_diagnostics, n_samples=n_samples, batch_size=batch_size, return_numpy=True
# )

In [ ]:
# # tarp_kwargs={"randoms_dependence": True, "randoms_dist": "normal", "randoms_scale": 0.001, "n_bootstrap": 120}
# tarp_kwargs = {"n_bootstrap": 100, "n_alpha_bins": 20}
# # tarp_kwargs.update({"randoms_dist": "normal", "randoms_scale": 0.1})
# # tarp_kwargs.update({"randoms_dependence": True})
# # tarp_kwargs.update({"randoms_dependence": True, "randoms_dist": "normal", "randoms_scale": 0.1})
# # tarp_kwargs.update({"randoms_dependence": False, "randoms_dist": "normal", "randoms_scale": 0.001})
# # tarp_kwargs = {}
# diagnostics.plot_tarp_check(
#     grid_preds_true, grid_preds_sample, grid_cosmos_diagnostics, **tarp_kwargs
# )

In [ ]:
# grid_preds_sample = model.plot_diagnostics(
#     # these must be the raw arrays where the cosmo and example axis are still separate
#     # grid_preds_true=file_dict["grid/pred"], 
#     # grid_cosmos=file_dict["grid/cosmo"],
#     grid_preds_true=grid_preds,
#     grid_cosmos=grid_cosmos,
#     n_samples=1_000,
#     n_cosmos=1_000,
#     # n_cosmos=5_000,
#     do_tarp=True,
#     batch_size=1000,
#     # tarp_kwargs={"randoms_scale": 0.1},
#     tarp_kwargs={"randoms_dependence": True, "randoms_dist": "normal", "randoms_scale": 0.001, "n_bootstrap": 120},
# )

# contours

## observations

### CosmoGrid internal

fiducial

In [ ]:
# obs_dict = {}

# shift = 0
# n_examples = 15
# i_fidu = 0
# range_examples = range(shift, n_examples+shift)

# for i_fidu in range(n_examples):
#     obs_dict[f"fiducial_{i_fidu}"] = {
#         "pred": fidu_preds[i_fidu], 
#         "point": {str(param): value for param, value in zip(params, parameters.get_fiducials(params, conf))},
#     }

# # obs_dict[f"fiducial_{i_fidu}"] = {
# #     "pred": fidu_preds[i_fidu], 
# #     "point": {str(param): value for param, value in zip(params, parameters.get_fiducials(params, conf))},
# # }

# # obs_dict[f"fiducial_mean_temp"] = {
# #     "pred": np.mean(fidu_preds[shift*n_examples:(1+shift)*n_examples], axis=0),
# #     "point": {str(param): value for param, value in zip(params, parameters.get_fiducials(params, conf))},
# # }

# # obs_dict[f"fiducial_mean_{str(range_examples).replace(' ', '')}"] = {
# #     "pred": np.mean([fidu_preds[i] for i in range_examples], axis=0), 
# #     "point": {str(param): value for param, value in zip(params, parameters.get_fiducials(params, conf))},
# # }

# # obs_dict[f"fiducial_stack_{str(range_examples).replace(' ', '')}"] = {
# #     "pred": np.stack([fidu_preds[i] for i in range_examples], axis=0), 
# #     "point": {str(param): value for param, value in zip(params, parameters.get_fiducials(params, conf))},
# # }

# # obs_dict[f"fiducial_mean"] = {
# #     "pred": np.mean(fidu_preds, axis=0),
# #     "point": {str(param): value for param, value in zip(params, parameters.get_fiducials(params, conf))},
# # }

# # obs_dict[f"fiducial_median"] = {
# #     "pred": np.median(fidu_preds, axis=0),
# #     "point": {str(param): value for param, value in zip(params, parameters.get_fiducials(params, conf))},
# # }

# # mse_deviation = np.mean((fidu_preds - np.median(fidu_preds, keepdims=True, axis=0))**2, axis=-1)
# # i_opt = np.argmin(mse_deviation)
# # print(f"i_opt = {i_opt}")
# # obs_dict[f"fiducial_opt_{i_opt}"] = {
# #     "pred": fidu_preds[i_opt], 
# #     "point": {str(param): value for param, value in zip(params, parameters.get_fiducials(params, conf))},
# # }

# # obs_dict[f"fiducial_stack_[{range_examples[0]}:{range_examples[-1]}]"] = {
# #     "pred": np.stack([fidu_preds[i] for i in range_examples], axis=0), 
# #     "point": {str(param): value for param, value in zip(params, parameters.get_fiducials(params, conf))},
# # }


grid 

In [ ]:
# obs_dict = {}

# shift = 5
# # n_examples = 4
# n_examples = 16
# i_fidu = 0

# for i_grid in range(n_examples):
#     i_grid *= 80
#     obs_dict[f"grid_{i_grid}"] = {
#         "pred": grid_preds[i_grid],
#         "point": {str(param): value for param, value in zip(params, grid_cosmos[i_grid])},
#     }

# # i_grid = 99
# # obs_dict[f"grid_{i_grid}"] = {
# #     "pred": grid_preds[i_grid],
# #     "point": {str(param): value for param, value in zip(params, grid_cosmos[i_grid])},
# # }

separate

In [ ]:
# obs_dict = {}
# mock_cosmo = {str(param): value for param, value in zip(params, parameters.get_fiducials(params, conf))}

# # mock_labels = ["bench_fidu"]
# # mock_labels += ["bench_box", "bench_particle", "bench_redshift"]
# # mock_labels += ["source_clustering_bgs_low", "source_clustering_bgs_high"]
# # mock_labels = ["ia_shell"]

# # mock_labels = ["Aia=0", "Aia=1", "Aia=-1"]
# # mock_labels = ["fidu_bary", "fidu_dmo"]
# # mock_labels = ["bench_fidu", "bench_fidu_seed=1234"]
# # mock_labels = ["bench_fidu_dmo"]

# # mock_labels = ["bench_fidu", "reduced_shear"]

# # mock_labels = ["ia_bin", "ia_shell"]

# mock_labels = ["no_mag", "mag"]
# # mock_labels = ["sys_ref", "sys_cont"]

# for mock_label in mock_labels:
#     if mock_label == "Aia=0":
#         mock_cosmo["Aia"] = 0.0
#         mock_cosmo["n_Aia"] = 0.0
#     elif mock_label == "Aia=1":
#         mock_cosmo["Aia"] = 1.0
#         mock_cosmo["n_Aia"] = 0.0
#     elif mock_label == "Aia=-1":
#         mock_cosmo["Aia"] = -1.0
#         mock_cosmo["n_Aia"] = 0.0
#     elif "ia_" in mock_label:
#         mock_cosmo["Aia"] = 1.0
#         mock_cosmo["n_Aia"] = 1.0

#     mock_preds = file_dict[f"mocks/pred/{mock_label}"]
    
#     obs_dict[f"{mock_label}_mean"] = {
#         "pred": np.mean(mock_preds, axis=0), 
#         "point": mock_cosmo,
#     }
    
#     # obs_dict[f"{mock_label}_[0,15]_stack"] = {
#     #     "pred": mock_preds[0:15], 
#     #     "point": mock_cosmo,
#     # }

In [ ]:
# # Aia tests
# mock_cosmo = {}
# mock_label = "cosmo_114996"
# metainfo = np.load("/global/homes/a/athomsen/multiprobe-simulation-forward-model/data/CosmoGridV11_original_metainfo.npy")
# for param in ["Om", "s8", "w0"]:
#     mock_cosmo[param] = metainfo[param][metainfo["sobol_index"] == int(mock_label[-6:])][0]
# mock_cosmo["n_Aia"] = 0.0
# mock_cosmo["bta"] = 0.0

# obs_dict = {}

# # mock_labels = ["cosmo_114996_Aia=0"]
# mock_labels = ["cosmo_114996_Aia=0", "cosmo_114996_Aia=1", "cosmo_114996_Aia=-1"]

# for mock_label in mock_labels:
#     if "Aia=0" in mock_label:
#         mock_cosmo["Aia"] = 0.0
#     elif "Aia=1" in mock_label:
#         mock_cosmo["Aia"] = 1.0
#     elif "Aia=-1" in mock_label:
#         mock_cosmo["Aia"] = -1.0

#     print(mock_cosmo)

#     mock_preds = file_dict[f"mocks/pred/{mock_label}"]
    
#     obs_dict[f"{mock_label}_mean"] = {
#         "pred": np.mean(mock_preds, axis=0), 
#         "point": mock_cosmo.copy(),
#     }
    
#     # obs_dict[f"{mock_label}_[0,15]_stack"] = {
#     #     "pred": mock_preds[0:15], 
#     #     "point": mock_cosmo,
#     # }

In [ ]:
# obs_dict = {}

# n_examples = 15

# # fiducial
# mock_label = "fiducial"
# mock_preds = file_dict[f"mocks/pred/{mock_label}"]
# mock_cosmo = {str(param): value for param, value in zip(params, parameters.get_fiducials(params, conf))}

# obs_dict[f"{mock_label}_mean"] = {
#     "pred": np.mean(mock_preds, axis=0), 
#     "point": mock_cosmo,
# }

# for shift in [15]:
#     range_examples = range(shift, n_examples+shift)
    
#     obs_dict[f"{mock_label}_[{shift},{shift+n_examples}]_mean"] = {
#         "pred": np.mean(mock_preds[range_examples], axis=0), 
#         "point": mock_cosmo,
#     }
    
#     obs_dict[f"{mock_label}_[{shift},{shift+n_examples}]_stack"] = {
#         "pred": mock_preds[range_examples], 
#         "point": mock_cosmo,
#     }

#     for i in range_examples:
#         obs_dict[f"{mock_label}_{i}"] = {
#             "pred": mock_preds[i], 
#             "point": mock_cosmo,
#         }


In [ ]:
# obs_dict = {}

# shift = 0
# n_examples = 15
# range_examples = range(shift, n_examples+shift)

# # grid
# # mock_label = "cosmo_000001"
# mock_label = "cosmo_114996"
# mock_preds = file_dict[f"mocks/pred/{mock_label}"]
# mock_cosmo = {str(param): value for param, value in zip(params, parameters.get_fiducials(params, conf))}
# metainfo = np.load("/global/homes/a/athomsen/multiprobe-simulation-forward-model/data/CosmoGridV11_original_metainfo.npy")
# for param in ["Om", "s8", "w0"]:
#     mock_cosmo[param] = metainfo[param][metainfo["sobol_index"] == int(mock_label[-6:])][0]

# # for i in range(3):
# #     obs_dict[f"{mock_label}_{i}"] = {
# #         "pred": mock_preds[i], 
# #         "point": mock_cosmo,
# #     }

# # obs_dict[f"{mock_label}_mean"] = {
# #     "pred": np.mean(mock_preds, axis=0), 
# #     "point": mock_cosmo,
# # }

# for shift in [45, 60]:
#     range_examples = range(shift, n_examples+shift)
#     obs_dict[f"{mock_label}_[{shift},{shift+n_examples}]_mean"] = {
#         "pred": np.mean(mock_preds[range_examples], axis=0), 
#         "point": mock_cosmo,
#     }
    
#     obs_dict[f"{mock_label}_[{shift},{shift+n_examples}]_stack"] = {
#         "pred": mock_preds[range_examples], 
#         "point": mock_cosmo,
#     }

### external

In [ ]:
obs_dict = {}

suffix = ""
# suffix = "_no_shear_weight"

buzzard_stack = []
buzzard_indices = list(range(0, 16))
buzzard_indices.remove(1)
# buzzard_indices.remove(10)
# buzzard_indices = [0, 2, 3, 4]
for i in buzzard_indices:
    pred = np.squeeze(file_dict[f"mocks/pred/Buzzard_{i}" + suffix])
    buzzard_stack.append(pred)
    print(pred)

    # obs_dict[f"Buzzard_{i}" + suffix] = {
    #     "pred": pred,
    #     "point": buzzard_cosmo,
    # }

buzzard_stack = np.stack(buzzard_stack, axis=0)
    
obs_dict[f"Buzzard_mean" + suffix] = {
    "pred": np.mean(buzzard_stack, axis=0),
    "point": buzzard_cosmo,
}

# obs_dict[f"Buzzard_stack" + suffix] = {
#     "pred": np.stack(buzzard_stack, axis=0),
#     "point": buzzard_cosmo,
# }

In [ ]:
# obs_dict = {}

# bias_dict = {"bg1": np.nan, "bg2": np.nan, "bg3": np.nan, "bg4": np.nan}
# # bias_dict = {"bg1": np.nan, "bg2": np.nan, "bg3": np.nan, "bg4": np.nan, "qbg1": np.nan, "qbg2": np.nan, "qbg3": np.nan, "qbg4": np.nan}

# obs_dict["Buzzard"] = {
#     "pred": np.squeeze(file_dict["mocks/pred/Buzzard"]),
#     "point": {"Om": 0.286, "s8": 0.82, "w0": -1, **bias_dict},
# }

# obs_dict["Cardinal"] = {
#     "pred": np.squeeze(file_dict["mocks/pred/Cardinal"]),
#     "point": {"Om": 0.286, "s8": 0.82, "w0": -1, **bias_dict},
# }

# # buzzard_flock_dir = "/global/u2/a/athomsen/multiprobe-simulation-forward-model/data/mock_observations/Buzzard_flock"
# # buzzard_flock_labels = os.listdir(buzzard_flock_dir)
# # buzzard_flock_labels = [file[23:-3] for file in buzzard_flock_labels]

# # buzzard_flock_preds = []
# # for i, buzzard_flock_label in enumerate(buzzard_flock_labels):
# #     buzzard = np.squeeze(file_dict[f"mocks/pred/{buzzard_flock_label}"])
# #     buzzard_flock_preds.append(buzzard)
    
# #     obs_dict[f"Buzzard_{i}"] = {
# #         "pred": buzzard,
# #         "point": {"Om": 0.286, "s8": 0.82, "w0": -1, **bias_dict},
# #     }
    
# # buzzard_flock_stack = np.stack(buzzard_flock_preds, axis=0)
# # obs_dict["Buzzard_flock_stack"] = {
# #     "pred": buzzard_flock_stack,
# #     "point": {"Om": 0.286, "s8": 0.82, "w0": -1, **bias_dict},
# # }

# # buzzard_flock_mean = np.mean(buzzard_flock_preds, axis=0)
# # obs_dict["Buzzard_flock_mean"] = {
# #     "pred": buzzard_flock_mean,
# #     "point": {"Om": 0.286, "s8": 0.82, "w0": -1, **bias_dict},
# # }

Octant footprint

In [ ]:
# obs_dict["MICE"] = {
#     "pred": np.squeeze(file_dict["mocks/pred/MICE"]),
#     "point": {"Om": 0.25, "s8": 0.8, "w0": -1, "bg": np.nan, "n_bg": np.nan},
# }

# obs_dict["Euclid"] = {
#     "pred": np.squeeze(file_dict["mocks/pred/Euclid"]),
#     "point": {"Om": 0.319, "s8": 0.83, "w0": -1, "bg": np.nan, "n_bg": np.nan},
# }

### MCMC and plotting

In [ ]:
extra_label = ""
# extra_label = "_42v"
# extra_label = "_no_i10"
# extra_label = "_walk_move"
# extra_label = "_walk_move_fewer_walkers_more_burnin"
# extra_label = f"_{n_examples}_mocks"
# extra_label = f"_9_mocks"
# extra_label = f"_{n_examples}_mocks_shift_{shift}"

for key in obs_dict.keys():
    print(f"\nStarting with mock observation {key}")
    # print(obs_dict[key]["pred"])
    
    posterior_samples = model.sample_posterior(
        obs_dict[key]["pred"],
        label=key+extra_label,
        n_walkers=1024,
        # n_walkers=128,
        n_burnin_steps=1000,
        # n_burnin_steps=1,
        n_samples=1024*1000,
        # n_samples=1024*1000,
        # n_samples=128*3000,
    )

    model.plot_contours(
        posterior_samples,
        obs_point=obs_dict[key]["point"],
        obs_label=key,
        label=key+extra_label,
        with_des_chain=False,
        density=True,
    )

### sample the posterior directly

In [ ]:
# # extra_label = f"_direct_{n_examples}_mocks_shift_{shift}"
# extra_label = "_npe"

# for key in obs_dict.keys():
#     print(f"\nStarting with mock observation {key}")
#     # print(obs_dict[key]["pred"])
    
#     posterior_samples = model.sample_likelihood(obs_dict[key]["pred"][np.newaxis], n_samples=100_000, batch_size=None, return_numpy=True)
#     posterior_samples = np.squeeze(posterior_samples)
    
#     plotting.plot_chains(
#         posterior_samples,
#         params=params,
#         conf=conf,
#         # cosmetics
#         title=None,
#         colors=None,
#         plot_labels="samples",
#         scale_to_prior=True,
#         group_params=False,
#         tri_kwargs={},
#         density=True,
#         # cosmo
#         plot_obs=True,
#         obs_point=obs_dict[key]["point"],
#         obs_label=key,
#     )
    
#     samples_file = os.path.join(model.model_dir, f"chain_{key}{extra_label}.npy")
#     print(samples_file)
#     np.save(samples_file, posterior_samples)

# old